In [1]:
!pip install wittgenstein pandas scikit-learn

import pandas as pd
from sklearn.datasets import load_iris
import wittgenstein as lw

# Load Iris dataset
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

df = pd.concat([X, y.rename("target")], axis=1)
print(df.head())

# RIPPER Algorithm
model = lw.RIPPER()
model.fit(X, y, pos_class=0)

print("\nRIPPER Rules:")
print(model.ruleset_)

# FOIL - Binary classification
df["target"] = (df["target"] == 0).astype(int)
attributes = list(X.columns)

def foil_gain(pb, nb, pa, na):
    if pa == 0 or pa + na == 0:
        return -1e9
    return pa * (pa/(pa+na) - pb/(pb+nb))

def foil(df):
    rules = []

    while df["target"].sum() > 0:
        covered = df.copy()
        rule = []

        while (covered["target"] == 0).sum() > 0:
            best = None

            for attr in attributes:
                for val in df[attr].unique():
                    s = covered[covered[attr] == val]
                    pa = s["target"].sum()
                    na = len(s) - pa
                    gain = foil_gain(
                        covered["target"].sum(),
                        len(covered) - covered["target"].sum(),
                        pa, na
                    )

                    if best is None or gain > best[0]:
                        best = (gain, attr, val, s)

            if best is None:
                break

            _, attr, val, covered = best
            rule.append((attr, val))

        rules.append(rule)
        df = df.drop(covered.index)

    return rules

rules = foil(df)

print("\nFOIL Rules:")
for rule in rules:
    conditions = " AND ".join(f"{a}={v}" for a, v in rule)
    print("IF", conditions, "THEN class=1")

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=99038 sha256=8b650c3d70dc8f539f91a16a97afd4895f2afec2904a5ab1906e146665f6337d
  Stored in directory: c:\users\student\appdata\local\pip\cache\wheels\ed\c4\1c\36ab0b352b6fc249d4bb154081713e3a42c868994e8aaa74de
Successfully built wittgenstein
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       